In [2]:
# 01 — Inspect Current-State Silver
#
# Purpose:
# Read the validated Silver current-state table that will feed
# the Gold analytical layer.
#
# Why:
# Gold should be built from the trusted, deduplicated Silver layer,
# not directly from raw Bronze data.

silver_table_name = "silver_sales_current"

silver_df = spark.table(silver_table_name)

print("=== Silver Source for Gold ===")
print(f"Source table: {silver_table_name}")
print(f"Row count: {silver_df.count()}")

silver_df.printSchema()

display(silver_df.limit(10))

StatementMeta(, 16ddffcf-3ecc-4169-917a-2f9be636e19a, 4, Finished, Available, Finished, False)

=== Silver Source for Gold ===
Source table: silver_sales_current
Row count: 9994
root
 |-- row_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- ship_date: date (nullable = true)
 |-- ship_mode: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- segment: string (nullable = true)
 |-- country: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- postal_code: string (nullable = true)
 |-- region: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- sub_category: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- sales: decimal(18,2) (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- discount: decimal(18,4) (nullable = true)
 |-- profit: decimal(18,2) (nullable = true)
 |-- source_file_name: string (nullable = true)


SynapseWidget(Synapse.DataFrame, 702809b2-45e2-4eae-a88b-61e334d7f23b)

In [3]:
from pyspark.sql import functions as F

# ============================================================
# CELL 02 — PREPARE GOLD SALES FACT
# ============================================================
# Purpose:
#   Select only the business fields required for the Gold sales
#   fact table and create a deterministic geography key.
#
# Why:
#   The Gold fact represents the transactional sales grain:
#   one row per unique Row ID.
#
#   Customer Name and Postal Code are intentionally excluded
#   from the reporting layer because they are personal data and
#   are not required for the regional sales analysis.
#
#   A geo_key is created because Region alone is not unique in
#   the Geography dimension. The combination of Region + State
#   + City identifies a unique geographic location.
# ============================================================

fact_df = (
    silver_df
    .select(
        "row_id",
        "order_id",
        "order_date",
        "ship_date",
        "ship_mode",
        "customer_id",
        "segment",
        "product_id",
        "region",
        "state",
        "city",
        "category",
        "sub_category",
        "product_name",
        "sales",
        "quantity",
        "discount",
        "profit"
    )

    # Create a deterministic key for each Region + State + City
    # combination.
    #
    # sha2(..., 256) produces the same key whenever the same
    # geographic combination is encountered again. This allows
    # the fact table and Geography dimension to be joined using
    # a stable key.
    .withColumn(
        "geo_key",
        F.sha2(
            F.concat_ws(
                "||",
                F.coalesce(F.col("region"), F.lit("")),
                F.coalesce(F.col("state"), F.lit("")),
                F.coalesce(F.col("city"), F.lit(""))
            ),
            256
        )
    )
)

# Display the prepared fact data for validation.
display(fact_df.limit(10))

# Basic validation checks.
print("=== Gold Sales Fact Preparation ===")
print(f"Rows prepared: {fact_df.count()}")
print(f"Columns: {len(fact_df.columns)}")
print(f"geo_key populated: {fact_df.filter(F.col('geo_key').isNotNull()).count()}")

StatementMeta(, 16ddffcf-3ecc-4169-917a-2f9be636e19a, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c2604ab7-4e33-4d82-89a8-f6fff8756ebb)

=== Gold Sales Fact Preparation ===
Rows prepared: 9994
Columns: 19
geo_key populated: 9994


In [6]:
# 03 — Create Gold Sales Fact Table
#
# Purpose:
# Persist the reporting-safe sales dataset as a Delta table
# in the Gold layer.
#
# Idempotency:
# The table is rebuilt from the current Silver state so that
# rerunning this Gold transformation produces the same result
# instead of appending duplicate records.
#
# Why overwrite:
# Silver already represents the authoritative current state.
# Gold should therefore be a reproducible presentation of that state.

gold_fact_table_name = "gold_sales_fact"

(
    fact_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(gold_fact_table_name)
)

print("=== Gold Sales Fact Created ===")
print(f"Table: {gold_fact_table_name}")
print(f"Rows written: {fact_df.count()}")
print(f"Columns: {len(fact_df.columns)}")

StatementMeta(, d917536a-03df-46ab-a0e3-a7709b00ae5c, 8, Finished, Available, Finished, False)

=== Gold Sales Fact Created ===
Table: gold_sales_fact
Rows written: 9994
Columns: 18


In [7]:
# 04 — Create Gold Date Dimension
#
# Purpose:
# Create a reusable calendar dimension for time-based analysis.
#
# Grain:
# One row per unique Order Date in the current Silver dataset.
#
# Why:
# A dedicated date dimension provides consistent Year, Quarter,
# Month and Date attributes for Power BI analysis.

from pyspark.sql import functions as F

date_dim_df = (
    silver_df
    .select(F.col("order_date").alias("date"))
    .where(F.col("order_date").isNotNull())
    .distinct()
    .withColumn("date_key", F.date_format("date", "yyyyMMdd").cast("int"))
    .withColumn("year", F.year("date"))
    .withColumn("quarter", F.quarter("date"))
    .withColumn("month", F.month("date"))
    .withColumn("month_name", F.date_format("date", "MMMM"))
    .withColumn("year_month", F.date_format("date", "yyyy-MM"))
    .select(
        "date_key",
        "date",
        "year",
        "quarter",
        "month",
        "month_name",
        "year_month"
    )
    .orderBy("date")
)

print("=== Gold Date Dimension ===")
print(f"Rows: {date_dim_df.count()}")
print(f"Columns: {len(date_dim_df.columns)}")

display(date_dim_df.limit(20))

StatementMeta(, d917536a-03df-46ab-a0e3-a7709b00ae5c, 9, Finished, Available, Finished, False)

=== Gold Date Dimension ===
Rows: 1238
Columns: 7


SynapseWidget(Synapse.DataFrame, a401610d-ecaa-4f9e-b09a-1abf17d2fe3e)

In [8]:
# 05 — Persist Gold Date Dimension
#
# Purpose:
# Save the prepared date dimension as a reusable Gold Delta table.
#
# Idempotency:
# The dimension is rebuilt from the current Silver state and
# overwritten on rerun, preventing duplicate date records.

gold_date_table_name = "gold_dim_date"

(
    date_dim_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(gold_date_table_name)
)

print("=== Gold Date Dimension Created ===")
print(f"Table: {gold_date_table_name}")
print(f"Rows written: {date_dim_df.count()}")
print(f"Columns: {len(date_dim_df.columns)}")

StatementMeta(, d917536a-03df-46ab-a0e3-a7709b00ae5c, 10, Finished, Available, Finished, False)

=== Gold Date Dimension Created ===
Table: gold_dim_date
Rows written: 1238
Columns: 7


In [9]:
# 06 — Prepare Gold Product Dimension
#
# Purpose:
# Create a reusable product dimension for product, category,
# and sub-category analysis.
#
# Grain:
# One row per unique Product ID.
#
# Why:
# Product attributes belong in a dimension rather than being
# repeatedly stored as descriptive attributes in analytical queries.

product_dim_df = (
    silver_df
    .select(
        "product_id",
        "product_name",
        "category",
        "sub_category"
    )
    .dropDuplicates(["product_id"])
    .orderBy("product_id")
)

print("=== Gold Product Dimension ===")
print(f"Rows: {product_dim_df.count()}")
print(f"Columns: {len(product_dim_df.columns)}")
print(f"Distinct Product IDs: {product_dim_df.select('product_id').distinct().count()}")

display(product_dim_df.limit(20))

StatementMeta(, d917536a-03df-46ab-a0e3-a7709b00ae5c, 11, Finished, Available, Finished, False)

=== Gold Product Dimension ===
Rows: 1862
Columns: 4
Distinct Product IDs: 1862


SynapseWidget(Synapse.DataFrame, e0dd7760-1042-48cb-8b51-4a8239cf8483)

In [10]:
# 07 — Persist Gold Product Dimension
#
# Purpose:
# Save the prepared product dimension as a reusable Gold Delta table.
#
# Idempotency:
# The dimension is rebuilt from the current Silver state and
# overwritten on rerun, preventing duplicate dimension records.

gold_product_table_name = "gold_dim_product"

(
    product_dim_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(gold_product_table_name)
)

print("=== Gold Product Dimension Created ===")
print(f"Table: {gold_product_table_name}")
print(f"Rows written: {product_dim_df.count()}")
print(f"Columns: {len(product_dim_df.columns)}")

StatementMeta(, d917536a-03df-46ab-a0e3-a7709b00ae5c, 12, Finished, Available, Finished, False)

=== Gold Product Dimension Created ===
Table: gold_dim_product
Rows written: 1862
Columns: 4


In [5]:
from pyspark.sql import functions as F

# ============================================================
# CELL 08 — CREATE GOLD GEOGRAPHY DIMENSION
# ============================================================
# Purpose:
#   Create the Geography dimension at Region + State + City
#   grain and generate a deterministic geo_key.
#
# Why:
#   Region is not unique because each region contains multiple
#   states and cities. Therefore, Region alone cannot be used
#   as the dimension key.
#
#   geo_key uniquely identifies each Region + State + City
#   combination and allows a proper one-to-many relationship
#   with the Gold Sales Fact in Power BI.
#
# Dimension grain:
#   One row per unique Region + State + City combination.
#
# Security:
#   Postal Code is excluded because it is personal data and
#   is not required for regional analysis.
# ============================================================

region_dim_df = (
    silver_df
    .select("region", "state", "city")
    .where(
        F.col("region").isNotNull()
        & F.col("state").isNotNull()
        & F.col("city").isNotNull()
    )
    .dropDuplicates(["region", "state", "city"])
    .withColumn(
        "geo_key",
        F.sha2(
            F.concat_ws(
                "||",
                F.coalesce(F.col("region"), F.lit("")),
                F.coalesce(F.col("state"), F.lit("")),
                F.coalesce(F.col("city"), F.lit(""))
            ),
            256
        )
    )
    .select("geo_key", "region", "state", "city")
    .orderBy("region", "state", "city")
)

# ============================================================
# VALIDATION
# ============================================================
# Confirm that each geography combination has exactly one
# unique geo_key.
# ============================================================

row_count = region_dim_df.count()

distinct_geo_key_count = (
    region_dim_df
    .select("geo_key")
    .distinct()
    .count()
)

print("=== Gold Geography Dimension ===")
print(f"Rows: {row_count}")
print(f"Columns: {len(region_dim_df.columns)}")
print(f"Distinct geo_keys: {distinct_geo_key_count}")

display(region_dim_df.limit(20))

StatementMeta(, 16ddffcf-3ecc-4169-917a-2f9be636e19a, 7, Finished, Available, Finished, False)

=== Gold Geography Dimension ===
Rows: 604
Columns: 4
Distinct geo_keys: 604


SynapseWidget(Synapse.DataFrame, e39b6a10-b7da-423e-a50f-21b298ce8a32)

In [6]:
# ============================================================
# CELL 09 — WRITE GOLD GEOGRAPHY DIMENSION
# ============================================================
# Purpose:
#   Persist the prepared Geography dimension as a Delta table
#   in the Gold layer.
#
# Why:
#   The Gold layer contains curated, analytics-ready data.
#   Persisting the Geography dimension allows Power BI and
#   other consumers to reuse the same governed geography data.
#
# Table grain:
#   One row per unique Region + State + City combination.
#
# Key:
#   geo_key uniquely identifies each geography combination.
#
# Rerun behavior:
#   The table is rebuilt using overwrite because the Geography
#   dimension is derived from the current Silver state.
#   overwriteSchema ensures the new geo_key column is reflected
#   in the Delta table schema.
# ============================================================

gold_region_table_name = "gold_dim_region"

(
    region_dim_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(gold_region_table_name)
)

print("=== Gold Geography Dimension Created ===")
print(f"Table: {gold_region_table_name}")
print(f"Rows written: {region_dim_df.count()}")
print(f"Columns: {len(region_dim_df.columns)}")

StatementMeta(, 16ddffcf-3ecc-4169-917a-2f9be636e19a, 8, Finished, Available, Finished, False)

=== Gold Geography Dimension Created ===
Table: gold_dim_region
Rows written: 604
Columns: 4


In [9]:
# ============================================================
# CELL — ADD GEO_KEY TO EXISTING GOLD SALES FACT
# ============================================================
# Purpose:
#   Add the new geo_key column to the existing Gold Sales Fact
#   only when the column does not already exist.
#
# Why:
#   The fact table already contains the correct current-state
#   rows and the SQL RLS policy is attached to this table.
#
#   The notebook may be rerun by the Master Pipeline, so the
#   schema change must be idempotent.
#
# Important:
#   Do not overwrite/recreate the table because doing so could
#   unnecessarily replace the existing table object and its
#   SQL RLS security configuration.
# ============================================================

# Check the existing Gold fact schema before attempting the
# ALTER TABLE operation.
gold_fact_columns = spark.table("gold_sales_fact").columns

if "geo_key" not in gold_fact_columns:

    # Add geo_key only on the first execution.
    # SHA-256 produces a hexadecimal string, so STRING is used.
    spark.sql("""
        ALTER TABLE gold_sales_fact
        ADD COLUMNS (geo_key STRING)
    """)

    print("=== geo_key Column Added ===")
    print("Column: geo_key")
    print("Data type: STRING")

else:

    # On reruns, the column already exists, so skip the schema
    # modification and allow the pipeline to continue safely.
    print("=== geo_key Column Already Exists ===")
    print("Skipping ALTER TABLE.")
    print("Gold fact schema is already ready for geo_key.")

StatementMeta(, 16ddffcf-3ecc-4169-917a-2f9be636e19a, 11, Finished, Available, Finished, False)

=== geo_key Column Added ===
Column: geo_key
Data type: STRING


In [10]:
# ============================================================
# CELL — POPULATE GEO_KEY IN GOLD SALES FACT
# ============================================================
# Purpose:
#   Populate the newly added geo_key column using the validated
#   geo_key values already calculated in fact_df.
#
# Why:
#   fact_df was created from the current Silver data and contains
#   the deterministic key generated from:
#
#       Region + State + City
#
#   Row ID is used to match the in-memory fact data to the
#   existing Gold fact because Row ID is the business key and
#   uniquely identifies each sales record in the curated data.
#
# Rerun behavior:
#   MERGE updates the geo_key for matching Row IDs and does not
#   create duplicate sales records.
# ============================================================

# Make the prepared fact DataFrame available to Spark SQL.
fact_df.createOrReplaceTempView("fact_geo_update")

# Update the existing Gold fact with the validated geo_key.
spark.sql("""
    MERGE INTO gold_sales_fact AS target
    USING fact_geo_update AS source
    ON target.row_id = source.row_id

    WHEN MATCHED THEN
        UPDATE SET
            target.geo_key = source.geo_key
""")

print("=== Gold Sales Fact geo_key Populated ===")

StatementMeta(, 16ddffcf-3ecc-4169-917a-2f9be636e19a, 12, Finished, Available, Finished, False)

=== Gold Sales Fact geo_key Populated ===


In [11]:
# ============================================================
# VALIDATION — VERIFY GEO_KEY IN GOLD SALES FACT
# ============================================================
# Confirm that:
#   1. The fact row count is unchanged.
#   2. geo_key exists.
#   3. Every fact row has a geo_key.
#   4. Row IDs remain unique.
# ============================================================

gold_fact_final_df = spark.table("gold_sales_fact")

fact_row_count = gold_fact_final_df.count()

geo_key_count = (
    gold_fact_final_df
    .filter(F.col("geo_key").isNotNull())
    .count()
)

duplicate_row_id_count = (
    gold_fact_final_df
    .groupBy("row_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("=== Gold Sales Fact Validation ===")
print(f"Rows: {fact_row_count}")
print(f"Columns: {len(gold_fact_final_df.columns)}")
print(f"Rows with geo_key: {geo_key_count}")
print(f"Duplicate Row IDs: {duplicate_row_id_count}")

gold_fact_final_df.printSchema()

StatementMeta(, 16ddffcf-3ecc-4169-917a-2f9be636e19a, 13, Finished, Available, Finished, False)

=== Gold Sales Fact Validation ===
Rows: 9994
Columns: 19
Rows with geo_key: 9994
Duplicate Row IDs: 0
root
 |-- row_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- ship_date: date (nullable = true)
 |-- ship_mode: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- segment: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- region: string (nullable = true)
 |-- state: string (nullable = true)
 |-- city: string (nullable = true)
 |-- category: string (nullable = true)
 |-- sub_category: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- sales: decimal(18,2) (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- discount: decimal(18,4) (nullable = true)
 |-- profit: decimal(18,2) (nullable = true)
 |-- geo_key: string (nullable = true)



In [12]:
# ============================================================
# GEO KEY RELATIONSHIP VALIDATION
# ============================================================
# Purpose:
#   Verify that every geo_key used by the Sales Fact exists in
#   the Geography Dimension.
#
# Why:
#   This confirms that the Gold fact and Geography dimension
#   can be safely related using geo_key in Power BI.
#
# Expected result:
#   Unmatched fact rows = 0
# ============================================================

fact_geo_keys = (
    spark.table("gold_sales_fact")
    .select("geo_key")
    .where(F.col("geo_key").isNotNull())
    .distinct()
)

region_geo_keys = (
    spark.table("gold_dim_region")
    .select("geo_key")
    .where(F.col("geo_key").isNotNull())
    .distinct()
)

unmatched_fact_geo_keys = (
    fact_geo_keys
    .join(region_geo_keys, on="geo_key", how="left_anti")
)

print("=== Geo Key Relationship Validation ===")
print(f"Distinct Fact geo_keys: {fact_geo_keys.count()}")
print(f"Distinct Dimension geo_keys: {region_geo_keys.count()}")
print(f"Unmatched Fact geo_keys: {unmatched_fact_geo_keys.count()}")

display(unmatched_fact_geo_keys.limit(10))

StatementMeta(, 16ddffcf-3ecc-4169-917a-2f9be636e19a, 14, Finished, Available, Finished, False)

=== Geo Key Relationship Validation ===
Distinct Fact geo_keys: 604
Distinct Dimension geo_keys: 604
Unmatched Fact geo_keys: 0


SynapseWidget(Synapse.DataFrame, 930cb30e-8b6a-4780-a4df-f402534b989d)

In [1]:
# ============================================================
# 10 — Create Regional Security Mapping
# ============================================================
# Purpose:
# Map each regional manager's user principal to the region
# they are authorized to access.
#
# Grain:
# One row per user-region assignment.
#
# Security design:
# This mapping will be used by the data-layer RLS policy.
# The user's identity will be matched to user_principal,
# and the mapped region will determine which sales rows
# the user is allowed to see.
#
# NOTE:
# These are demonstration identities. Replace them with the
# actual Entra ID UPNs used for the assessment demonstration.

security_mapping_data = [
    ("manager.central@cargills-demo.com", "Central"),
    ("manager.east@cargills-demo.com", "East"),
    ("manager.west@cargills-demo.com", "West"),
    ("it23599468@my.sliit.lk", "East")

]

security_user_region_df = spark.createDataFrame(
    security_mapping_data,
    ["user_principal", "region"]
)

print("=== Regional Security Mapping ===")
print(f"Rows: {security_user_region_df.count()}")
print(f"Columns: {len(security_user_region_df.columns)}")

display(security_user_region_df)

StatementMeta(, 944cb5cc-7619-4fed-a85d-0c5d88ad8e88, 3, Finished, Available, Finished, False)

=== Regional Security Mapping ===
Rows: 4
Columns: 2


SynapseWidget(Synapse.DataFrame, 19505a1b-6f8b-4ca5-80e3-c1581581277f)

In [2]:
# ============================================================
# 11 — Persist Regional Security Mapping
# ============================================================
# Purpose:
# Persist the user-to-region authorization mapping as a Gold
# security table.
#
# Grain:
# One row per authorized user-region assignment.
#
# This table will be consumed by the data-layer security design.

security_table_name = "security_user_region"

(
    security_user_region_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(security_table_name)
)

print("=== Security Mapping Table Created ===")
print(f"Table: {security_table_name}")
print(f"Rows written: {security_user_region_df.count()}")
print(f"Columns: {len(security_user_region_df.columns)}")

StatementMeta(, 944cb5cc-7619-4fed-a85d-0c5d88ad8e88, 4, Finished, Available, Finished, False)

=== Security Mapping Table Created ===
Table: security_user_region
Rows written: 4
Columns: 2
